# 06 - Ablation Study

This notebook runs systematic ablation studies to evaluate component contributions.

## Ablation Experiments
1. Text-only model
2. Image-only model
3. Metadata-only model
4. Text + Image (no metadata)
5. Text + Metadata (no image)
6. Full model (baseline for comparison)
7. No cross-attention
8. No gating fusion
9. No modality dropout

# Load embeddings and data
text_embeddings = np.load(EMBEDDINGS_DIR / 'text' / 'all_embeddings.npy')
image_embeddings = np.load(EMBEDDINGS_DIR / 'image' / 'all_embeddings.npy')
metadata_features = np.load(EMBEDDINGS_DIR / 'metadata' / 'all_features.npy')
id_mapping = pd.read_csv(EMBEDDINGS_DIR / 'id_mapping.csv')

# Create tensors
text_tensor = torch.from_numpy(text_embeddings).float().to(device)
image_tensor = torch.from_numpy(image_embeddings).float().to(device)
metadata_tensor = torch.from_numpy(metadata_features).float().to(device)
labels = torch.from_numpy(id_mapping['label'].values).float().to(device)

# Get indices
train_idx = id_mapping[id_mapping['split'] == 'train'].index.values
val_idx = id_mapping[id_mapping['split'] == 'val'].index.values
test_idx = id_mapping[id_mapping['split'] == 'test'].index.values

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# Base model
class TextOnlyModel(nn.Module):
    def __init__(self, input_dim=768):
        super().__init__()
        self.proj = nn.Linear(input_dim, 256)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
    
    def forward(self, x):
        feat = torch.relu(self.proj(x))
        logits = self.classifier(feat)
        return logits

class MultimodalBaseline(nn.Module):
    def __init__(self, text_dim=768, image_dim=768, metadata_dim=32, hidden_dim=256):
        super().__init__()
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.image_proj = nn.Linear(image_dim, hidden_dim)
        self.metadata_proj = nn.Linear(metadata_dim, hidden_dim)
        
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 3),
            nn.Softmax(dim=1)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
    
    def forward(self, text, image, metadata):
        text_feat = torch.relu(self.text_proj(text))
        image_feat = torch.relu(self.image_proj(image))
        metadata_feat = torch.relu(self.metadata_proj(metadata))
        
        concat_feat = torch.cat([text_feat, image_feat, metadata_feat], dim=1)
        gates = self.gate(concat_feat)
        
        fused = (gates[:, 0:1] * text_feat + 
                 gates[:, 1:2] * image_feat + 
                 gates[:, 2:3] * metadata_feat)
        
        logits = self.classifier(fused)
        return logits

print("Model variants defined")

def train_and_evaluate_model(model, train_loader, val_loader, test_data, num_epochs=10, model_name=""):
    """
    Train and evaluate a model variant
    """
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    
    best_val_loss = float('inf')
    best_epoch = 0
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        for batch in train_loader:
            if isinstance(batch, dict):
                loss_batch = list(batch.values())
            else:
                loss_batch = batch
            
            # Forward pass (simplified)
            if len(loss_batch) == 2:
                x, y = loss_batch
                logits = model(x)
            else:
                logits = model(*loss_batch[:-1])
                y = loss_batch[-1]
            
            loss = criterion(logits, y.unsqueeze(1) if logits.shape != y.shape else y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        if len(train_loader) > 0:
            train_loss /= len(train_loader)
    
    # Test evaluation (simplified)
    model.eval()
    with torch.no_grad():
        if len(test_data) == 2:
            x_test, y_test = test_data
            logits = model(x_test)
        else:
            logits = model(*test_data[:-1])
            y_test = test_data[-1]
        
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs > 0.5).astype(int).flatten()
        y_true = y_test.cpu().numpy().astype(int)
        
        metrics = {
            'accuracy': accuracy_score(y_true, preds),
            'precision': precision_score(y_true, preds, zero_division=0),
            'recall': recall_score(y_true, preds, zero_division=0),
            'f1': f1_score(y_true, preds, zero_division=0),
            'roc_auc': roc_auc_score(y_true, probs)
        }
    
    return metrics

print("Training function defined")

results = {}
batch_size = 32

# Prepare test data
test_text = text_tensor[test_idx]
test_image = image_tensor[test_idx]
test_metadata = metadata_tensor[test_idx]
test_labels = labels[test_idx]

# Create dummy DataLoaders for consistency
train_loader = [None] * (len(train_idx) // batch_size)
val_loader = [None] * (len(val_idx) // batch_size)

## Results Summary

## Visualizations

In [ ]:
# Plot F1 scores comparison
fig, ax = plt.subplots(figsize=(12, 6))
f1_scores = results_df['f1'].sort_values(ascending=False)
colors = ['#2ecc71' if score == f1_scores.max() else '#3498db' for score in f1_scores.values]
bars = ax.barh(range(len(f1_scores)), f1_scores.values, color=colors)
ax.set_yticks(range(len(f1_scores)))
ax.set_yticklabels(f1_scores.index)
ax.set_xlabel('F1-Score', fontsize=12)
ax.set_title('Ablation Study: F1-Score Comparison', fontsize=14, fontweight='bold')
ax.set_xlim([0, 1])

# Add value labels
for i, (idx, value) in enumerate(f1_scores.items()):
    ax.text(value + 0.02, i, f'{value:.4f}', va='center', fontsize=10, fontweight='bold')

ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'figures' / 'ablation_f1_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap of all metrics
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(results_df, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax,
            cbar_kws={'label': 'Score'}, vmin=0, vmax=1)
ax.set_title('Ablation Study: All Metrics Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'figures' / 'ablation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Radar chart for metrics comparison
from math import pi

categories = list(results_df.columns)
num_vars = len(categories)
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

# Plot for each model variant
for model_name in results_df.index:
    values = results_df.loc[model_name].values.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name)
    ax.fill(angles, values, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=10)
ax.set_ylim(0, 1)
ax.set_title('Ablation Study: Metrics Radar Chart', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
ax.grid(True)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'figures' / 'ablation_radar.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## Key Insights

In [ ]:
print("\n" + "="*70)
print("KEY INSIGHTS")
print("="*70)

# Best model
best_model = results_df['f1'].idxmax()
best_f1 = results_df['f1'].max()
print(f"\nBest Model: {best_model}")
print(f"Best F1-Score: {best_f1:.4f}")

# Modality contribution
if 'Text-only' in results_df.index and 'Full Model' in results_df.index:
    text_contribution = (results_df.loc['Full Model', 'f1'] - results_df.loc['Text-only', 'f1']) / results_df.loc['Text-only', 'f1'] * 100
    print(f"\nModality Contributions (relative to single modality):")
    print(f"  Text only F1: {results_df.loc['Text-only', 'f1']:.4f}")
    print(f"  Full model F1: {results_df.loc['Full Model', 'f1']:.4f}")
    if not np.isnan(text_contribution):
        print(f"  Multimodal improvement: {text_contribution:.1f}%")

print("\n" + "="*70)